In [1]:
!git clone https://github.com/jjeong3150/AIFFEL_final_pjt_book-recommendation-agent

fatal: destination path 'AIFFEL_final_pjt_book-recommendation-agent' already exists and is not an empty directory.


# Main — CRS × User Simulation 오케스트레이션

```
build_graph.ipynb  →  app, initial_state, config 제공 (while 루프 셀 제거 필요)
user_sim.ipynb     →  UserSimAgent, PERSONA_TEMPLATES
main.ipynb         →  Queue 기반으로 두 시스템 연결
```

**주의**: `build_graph.ipynb`의 마지막 셀 (while 루프)은 주석 처리하거나 삭제하세요.  
루프 로직은 이 노트북이 담당합니다.

## 1. 노트북 로드

In [2]:
# CRS 시스템 로드 (app, initial_state, config 가 네임스페이스로 올라옴)
%run /content/AIFFEL_final_pjt_book-recommendation-agent/src/pipeline/build_graph.ipynb

fatal: destination path 'AIFFEL_final_pjt_book-recommendation-agent' already exists and is not an empty directory.
fatal: destination path 'AIFFEL_final_pjt_book-recommendation-agent' already exists and is not an empty directory.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[LocalReranker] 캐시 로드: /content/drive/MyDrive/aiffel_final_pjt/models/BAAI_bge-reranker-v2-m3


/tmp/ipykernel_11692/896750007.py:41: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(


In [3]:
# User Sim 로드 (UserSimAgent, PERSONA_TEMPLATES 가 네임스페이스로 올라옴)
%run /content/AIFFEL_final_pjt_book-recommendation-agent/src/simulation/user_sim_test.ipynb

사용 가능한 페르소나: ['직장인_SF팬', '대학생_문학팬', '중년_역사_비문학']
UserSimAgent 정의 완료


## 2. 오케스트레이션 설정

In [4]:
import queue
import threading
import json
import copy
from langchain_core.messages import HumanMessage

# 통신 큐
user_to_crs = queue.Queue()   # UserSim → CRS (사용자 응답)
crs_to_user = queue.Queue()   # CRS → UserSim (질문 or 완료 신호)

# 결과 수집
eval_results = []

print("큐 초기화 완료")

큐 초기화 완료


## 3. CRS 실행 함수

`input()` 대신 `user_to_crs` 큐에서 읽어옵니다.

In [5]:
def _extract_ai_responses(state: dict[str, Any]) -> list[str]:
    """상태의 messages에서 마지막 사용자 메시지 이후의 AI 메시지들을 추출."""
    messages = state.get("messages", [])
    responses = []
    # 역순으로 순회하며 마지막 HumanMessage 이전의 AI 메시지들 수집
    for msg in reversed(messages):
        if isinstance(msg, HumanMessage) or getattr(msg, "type", None) == "human":
            break
        if isinstance(msg, AIMessage) or getattr(msg, "type", None) == "ai":
            responses.append(msg.content)
    responses.reverse()
    return responses


In [6]:
async def run_crs(thread_id: str):
    session_config = {"configurable": {"thread_id": thread_id}}
    state = copy.deepcopy(initial_state)
    result = await app.ainvoke(state, config=session_config)

    while True:
        snapshot = app.get_state(session_config)

        if snapshot.next == ():  # 그래프 완료
            crs_to_user.put({"__done__": True, "result": result})
            break

        # AI 질문 추출 후 UserSim에 전달
        ai_responses = _extract_ai_responses(result)
        if ai_responses:
            crs_to_user.put(ai_responses[-1])

        # UserSim 응답 대기
        user_input = user_to_crs.get()

        # 사용자 입력 주입 후 resume
        app.update_state(session_config, {"messages": [HumanMessage(content=user_input)]})
        result = await app.ainvoke(None, config=session_config)

print("run_crs 정의 완료")

run_crs 정의 완료


## 4. UserSim 실행 함수

In [7]:
def run_user_sim(persona: dict, result_collector: list):
    """
    UserSimAgent를 실행하며 CRS 질문에 자동 응답합니다.
    세션이 끝나면 결과를 result_collector에 추가합니다.
    """
    agent = UserSimAgent(persona=persona, verbose=True)

    while True:
        message = crs_to_user.get()  # CRS 메시지 대기 (블로킹)

        if isinstance(message, dict) and message.get("__done__"):
            crs_result = message["result"]
            result_collector.append({
                "persona":         persona,
                "user_profile":    crs_result.get("user_profile", {}),
                "summary":         crs_result.get("summary", ""),
                "recommendations": crs_result.get("recommendations", []),
                "final_message": (
                    crs_result["messages"][-1].content
                    if crs_result.get("messages") else ""
                ),
                "conversation": agent.get_history()
            })
            break

        response = agent.answer(message)
        user_to_crs.put(response)

print("run_user_sim 정의 완료")

run_user_sim 정의 완료


## 5. 단일 세션 실행

In [8]:
async def run_session(persona, eval_results, thread_id):
    t_user = threading.Thread(target=run_user_sim, args=(persona, eval_results))
    t_user.start()
    await run_crs(thread_id)
    t_user.join()

# 페르소나 선택
persona   = PERSONA_TEMPLATES["중년_역사_비문학"]
thread_id = "eval_session_001"

await run_session(persona, eval_results, thread_id)

print("\n" + "="*50)
print("세션 종료")
print("="*50)


[Turn 1]
  CSR  : 책을 읽고 싶어하는 이유가 궁금한데요, 혹시 특정한 목표나 동기가 있으신가요? 예를 들어, 지식을 쌓고 싶다거나, 여가를 즐기고 싶다거나 하는 것처럼요.
  USER : 주로 수업에 활용할 수 있는 흥미로운 역사 이야기를 찾고 싶어서 책을 읽어요. 학생들에게 생생하고 재미있는 내용을 전달해주고 싶거든요. 때로는 지식을 쌓으면서 저 스스로도 더 깊이 이해하고 싶기도 하고요.


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 29.8MiB/s]



[Turn 2]
  CSR  : 그렇군요! 학생들에게 흥미로운 내용을 전달하고 싶으시군요. 그러면 혹시 역사 관련 책의 난이도에 대해서는 어떤 편을 더 선호하시나요? 예를 들어, 좀 더 쉽게 읽히는 책을 원하시는지, 아니면 깊이 있는 내용을 다룬 책을 원하시는지 궁금합니다.
  USER : 학생들에게 전달하기 위해서는 쉽게 읽히는 책도 좋고, 깊이 있는 내용을 다룬 책도 모두 좋아요. 상황에 따라 적합한 난이도를 선택해요. 그래서 다양한 접근이 가능한 책들을 선호합니다.

[Turn 3]
  CSR  : 최근에 읽으신 역사 관련 책 중에서 학생들에게 흥미롭게 전달할 수 있을 만한 책이 있나요? 책 제목과 함께 간단한 소감도 공유해주시면 좋겠습니다. 어떤 이야기가 기억에 남았는지 궁금해요!
  USER : 최근에 읽은 "총, 균, 쇠"라는 책이 특히 흥미로웠어요. 인류의 발전과 다양한 문명의 차이를 설명하면서 학생들에게도 교훈이 될 만한 이야기가 많았죠. 특히 각 대륙의 환경 조건이 인류 역사에 미친 영향을 다룬 부분이 인상 깊어서 수업에 잘 활용할 수 있을 것 같아요.
추출된 장르: ['역사', '대학교재/전문서적', '자기계발']

[Query Transformations]
  [Step-back]  : 학생들에게 생생하고 흥미로운 역사 이야기를 전달하기 위한 다양한 난이도의 역사 서적을 찾아보세요. 인류 발전과 문명 차이를 탐구하며 수업에 활용할 수 있는 통찰을 제공하는 책들을 추천합니다.
  [Rewritten]  : 학생들에게 생생하고 흥미로운 역사 이야기를 전달할 수 있는 다양한 난이도의 역사 서적을 찾고 있습니다. 인류 발전과 문명 차이에 대한 깊이 있는 통찰을 제공하는 쉽고도 흥미로운 읽을거리를 원합니다.
  [Sub-queries]: ['학생들에게 생생하고 흥미로운 역사 이야기를 전달할 수 있는 다양한 난이도의 역사 서적을 찾고 있습니다.', '인류 발전과 문명 차이에 대한 깊이 있는 통찰을 제공하는 책을 원합니다.', '역사적 사건이나 인물에


세션 종료


## 6. 결과 확인

In [10]:
if eval_results:
    r = eval_results[-1]

    print("[페르소나]")
    print(json.dumps(r["persona"], ensure_ascii=False, indent=2))

    print("\n[추출된 프로필]")
    print(r["user_profile"])

    print("\n[요약]")
    print(r["summary"])

    print("\n[책 리스트]")
    print(r["recommendations"])

    print("\n[추천 결과]")
    print(r["final_message"])
else:
    print("결과 없음")

[페르소나]
{
  "age_group": "40대 중반",
  "job": "중학교 역사 교사",
  "favorite_genre": "역사, 교양 비문학",
  "reading_frequency": "한 달에 3~4권",
  "mood": "수업에 활용할 수 있는 흥미로운 역사 이야기",
  "disliked_genre": "공포, 오컬트",
  "reading_experience": "유발 하라리의 사피엔스를 인상 깊게 읽음"
}

[추출된 프로필]
reading_goal=ProfileSlot(value='주로 수업에 활용할 수 있는 흥미로운 역사 이야기를 찾고 싶어서 책을 읽어요. 학생들에게 생생하고 재미있는 내용을 전달해주고 싶거든요.', status=<SlotStatus.FILLED: 'filled'>, retry_count=0, MAX_RETRIES=3) preferred_genre=ProfileSlot(value='역사', status=<SlotStatus.FILLED: 'filled'>, retry_count=0, MAX_RETRIES=3) reading_style=ProfileSlot(value='저 스스로도 더 깊이 이해하고 싶기도 하다', status=<SlotStatus.FILLED: 'filled'>, retry_count=0, MAX_RETRIES=3) difficulty_level=ProfileSlot(value='다양한 난이도 선택', status=<SlotStatus.FILLED: 'filled'>, retry_count=0, MAX_RETRIES=3) current_context=ProfileSlot(value='학생들에게 생생하고 재미있는 내용을 전달하고 싶음', status=<SlotStatus.FILLED: 'filled'>, retry_count=0, MAX_RETRIES=3)

[요약]
사용자는 학생들에게 생생하고 흥미로운 역사 이야기를 전달하기 위해 책을 읽고 싶어합니다. 역사 분야에 관심이 많으며, 다양한 난이도의

## 7. 다중 페르소나 배치 평가

In [11]:
async def run_batch_evaluation(persona_dict: dict, base_thread_id: str = "batch"):
    global user_to_csr, csr_to_user
    batch_results = []

    for i, (name, persona) in enumerate(persona_dict.items()):
        print(f"\n{'='*50}")
        print(f"[{i+1}/{len(persona_dict)}] 페르소나: {name}")
        print(f"{'='*50}")

        user_to_csr = queue.Queue()
        csr_to_user = queue.Queue()

        session_results = []
        thread_id = f"{base_thread_id}_{i}"

        await run_session(persona, session_results, thread_id)

        if session_results:
            result = session_results[0]
            result["persona_name"] = name
            batch_results.append(result)
            print(f"  완료: 추천 {len(result.get('recommendations', []))}건")

    print(f"\n배치 평가 완료: {len(batch_results)}/{len(persona_dict)} 성공")
    return batch_results

# 실행 (주석 해제 시 전체 페르소나 평가)
# all_results = await run_batch_evaluation(PERSONA_TEMPLATES)